# Evaluation: decile analysis, lift charts, and campaign ROI

This notebook evaluates the calibrated propensity model through
decile analysis, lift and calibration charts, per-decile ROI calculations,
and campaign targeting recommendations. We interpret the AUC of 0.775
in the context of marketing campaign optimization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')

from sklearn.calibration import calibration_curve

from src.data_loader import prepare_full_pipeline, get_feature_columns
from src.model import (
    build_models, train_and_calibrate, evaluate_model,
    decile_analysis, print_decile_table, campaign_roi,
    plot_lift_chart, plot_decile_response, plot_calibration,
    plot_feature_importance
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Train the pipeline

In [ ]:
data = prepare_full_pipeline(path='../data/marketing_campaign.csv')
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
feature_names = data['feature_names']

raw_models = build_models(seed=42)
calibrated_models = train_and_calibrate(raw_models, X_train, y_train)

# Evaluate and select best
eval_results = {}
for name, model in calibrated_models.items():
    eval_results[name] = evaluate_model(name, model, X_test, y_test)

best_name = max(eval_results, key=lambda n: eval_results[n]['auc'])
best_prob = eval_results[best_name]['y_prob']
print(f'\nBest model: {best_name} (AUC={eval_results[best_name]["auc"]:.4f})')

## 2. Decile analysis

Decile analysis is the core deliverable for marketing. We rank all
customers by predicted propensity and divide them into 10 equal groups.
The top deciles should capture a disproportionate share of responders.

In [ ]:
decile_df = decile_analysis(y_test, best_prob)
print_decile_table(decile_df)

In [ ]:
# Response rate by decile bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1565C0' if d <= 2 else '#90CAF9' for d in decile_df['decile']]
ax.bar(decile_df['decile'].astype(int), decile_df['response_rate'] * 100,
       color=colors)
ax.axhline(y_test.mean() * 100, color='red', linestyle='--',
           label=f'Overall rate ({y_test.mean():.1%})')
ax.set_xlabel('Decile (0 = highest propensity)')
ax.set_ylabel('Response rate (%)')
ax.set_title('Response rate by propensity decile')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Lift chart

The lift chart shows what percentage of responders we capture by
contacting the top X% of customers (ranked by propensity). The
diagonal represents random targeting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pcts = np.arange(1, len(decile_df) + 1) / len(decile_df) * 100
cum_resp = decile_df['cumulative_response_pct'].values * 100

ax.plot(pcts, cum_resp, 'b-o', linewidth=2, markersize=8, label='Model')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Random targeting')
ax.fill_between(pcts, pcts, cum_resp, alpha=0.15, color='blue')

ax.set_xlabel('Percentage of customers contacted')
ax.set_ylabel('Cumulative percentage of responders captured')
ax.set_title('Cumulative lift chart')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate key points
ax.annotate(f'{cum_resp[2]:.0f}% captured\ncontacting 30%',
            xy=(30, cum_resp[2]), xytext=(45, cum_resp[2] - 15),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.tight_layout()
plt.show()

print(f'Contacting top 30% captures {cum_resp[2]:.0f}% of all responders')
print(f'Lift at 30%: {cum_resp[2] / 30:.2f}x vs random')

## 4. Calibration curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_map = {'Logistic Regression': '#2196F3', 'Random Forest': '#4CAF50', 'XGBoost': '#FF9800'}

for name, model in calibrated_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fraction_pos, mean_predicted = calibration_curve(y_test, y_prob, n_bins=10)
    ax.plot(mean_predicted, fraction_pos, 's-', label=name,
            color=colors_map.get(name, 'gray'), linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed frequency')
ax.set_title('Calibration curves')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. ROI calculation per decile

We calculate the expected revenue and ROI for targeting each decile,
using the assumptions from `model.py`: $2 cost per contact,
$15/month upsell value, 12-month customer lifetime.

In [ ]:
cost_per_contact = 2.0
upsell_monthly = 15.0
months = 12

roi_data = []
for _, row in decile_df.iterrows():
    n = row['n_customers']
    resp = row['n_responders']
    rate = row['response_rate']

    cost = n * cost_per_contact
    revenue = resp * upsell_monthly * months
    profit = revenue - cost
    roi = (revenue - cost) / cost * 100 if cost > 0 else 0

    roi_data.append({
        'decile': int(row['decile']),
        'customers': n,
        'responders': int(resp),
        'response_rate': f'{rate:.1%}',
        'cost': f'${cost:,.0f}',
        'revenue': f'${revenue:,.0f}',
        'profit': f'${profit:,.0f}',
        'roi': f'{roi:.0f}%',
    })

roi_table = pd.DataFrame(roi_data)
print('Per-decile ROI analysis:')
roi_table

In [ ]:
# Visualize per-decile profit
profits = []
for _, row in decile_df.iterrows():
    n = row['n_customers']
    resp = row['n_responders']
    profit = resp * upsell_monthly * months - n * cost_per_contact
    profits.append(profit)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4CAF50' if p > 0 else '#FF5722' for p in profits]
ax.bar(decile_df['decile'].astype(int), profits, color=colors)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Decile (0 = highest propensity)')
ax.set_ylabel('Profit ($)')
ax.set_title('Profit per decile (revenue minus contact cost)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Campaign targeting recommendations

We compare a mass campaign (contact everyone) vs targeted campaign
(contact top 3 deciles only) using the `campaign_roi()` function.

In [ ]:
roi_results = campaign_roi(decile_df, cost_per_contact=2.0,
                           upsell_monthly=15.0, months=12)

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Contacts
axes[0].bar(['Mass', 'Targeted'],
            [roi_results['mass_customers'], roi_results['targeted_customers']],
            color=['#90CAF9', '#1565C0'])
axes[0].set_title('Customers contacted')
axes[0].set_ylabel('Count')

# Response rate
axes[1].bar(['Mass', 'Targeted'],
            [roi_results['mass_rate'] * 100, roi_results['targeted_rate'] * 100],
            color=['#90CAF9', '#1565C0'])
axes[1].set_title('Response rate (%)')
axes[1].set_ylabel('Rate (%)')

# ROI
axes[2].bar(['Mass', 'Targeted'],
            [roi_results['mass_roi_pct'], roi_results['targeted_roi_pct']],
            color=['#90CAF9', '#1565C0'])
axes[2].set_title('ROI (%)')
axes[2].set_ylabel('ROI (%)')

plt.suptitle('Mass vs targeted campaign comparison', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Cost savings from targeting: ${roi_results["cost_savings"]:,.0f}')
print(f'Conversion lift: {roi_results["conversion_lift_pct"]:.1f}%')

## 7. AUC 0.775 in context

An AUC of ~0.775 is a strong result for marketing propensity scoring.
Here is why:

In [ ]:
auc = eval_results[best_name]['auc']

print(f'Model AUC: {auc:.3f}')
print()
print('Interpretation:')
print('  - AUC 0.50 = random guessing (no predictive value)')
print('  - AUC 0.60-0.70 = weak discrimination')
print('  - AUC 0.70-0.80 = acceptable for marketing applications')
print('  - AUC 0.80-0.90 = strong discrimination')
print('  - AUC > 0.90 = often indicates data leakage in marketing')
print()
print(f'At AUC {auc:.3f}, the model correctly ranks a random responder')
print(f'above a random non-responder {auc:.1%} of the time.')
print()
print('For upsell targeting, the practical value is in the lift:')
print(f'  - Top decile response rate is {decile_df.iloc[0]["response_rate"]:.1%}')
print(f'  - Overall response rate is {y_test.mean():.1%}')
print(f'  - Lift in top decile: {decile_df.iloc[0]["response_rate"] / y_test.mean():.1f}x')
print()
print('This lift enables targeted campaigns that are 2-3x more')
print('cost-efficient than untargeted outreach.')

## 8. Feature importance

In [ ]:
plot_feature_importance(
    calibrated_models[best_name], feature_names, top_n=15
)
plt.show()

## Summary

The propensity scoring model achieves AUC ~0.775 with good calibration.
Key findings:

- **Decile analysis** confirms the model concentrates responders in the
  top deciles, with the top 30% capturing the majority of responses
- **Lift chart** shows 2-3x improvement over random targeting
- **Per-decile ROI** is positive for the top 5-6 deciles and negative
  for the bottom deciles, defining a clear targeting cutoff
- **Targeted campaign** (top 3 deciles) achieves higher ROI and lower
  cost than a mass campaign while capturing most of the revenue
- **Previous upsell response** and **income** are the strongest predictors
- **Calibration** ensures predicted probabilities are trustworthy for
  threshold-based campaign decisions